### No-show Prediction - Logistic Regression

This notebook analyzes healthcare appointment data to predict patient no-shows. The workflow includes:

- Reading and loading the dataset
- Data cleaning and schema definition
- Handling missing values and dropping unnecessary columns
- Identifying and scaling numeric features for modeling

The goal is to prepare the data for building predictive models to identify factors influencing patient attendance.

### Read in scaled features

In [0]:
import os, yaml
config_file = "../config.yaml" if os.path.exists("../config.yaml") else "config.example.yaml"

with open(config_file) as f:
    cfg = yaml.safe_load(f)
train_scaled_path = cfg["train_scaled_dataset_path"]
test_scaled_path = cfg["test_scaled_dataset_path"]
volume_path = cfg["volume_path"]
logistic_regression_model_path = cfg["lr_model_path"]


train_scaled = spark.table(train_scaled_path)
test_scaled = spark.table(test_scaled_path)

# Split train into train/validation (75/25 of the 80% = 60/20 overall)
# Validation set is used for threshold tuning; test set is held out for final evaluation
train_scaled, val_scaled = train_scaled.randomSplit([0.75, 0.25], seed=42)

# train_scaled = spark.table("default.no_show_train_scaled_tbl")
# test_scaled = spark.table("default.no_show_test_scaled_tbl")

In [0]:
print(f"Features read in from Delta Table")
print(f"  Train: {train_scaled.count():,} rows, {len(train_scaled.columns)} columns")
print(f"  Val:   {val_scaled.count():,} rows, {len(val_scaled.columns)} columns")
print(f"  Test:  {test_scaled.count():,} rows, {len(test_scaled.columns)} columns")

#### Create Logistic Regression model

In [0]:
import os
from pyspark.ml.classification import LogisticRegression

# os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/default/no_show_volume"
os.environ["SPARKML_TEMP_DFS_PATH"] = volume_path

# Re-instantiate LogisticRegression with correct featuresCol
lr = LogisticRegression(
    featuresCol="selectedFeatures",
    labelCol="Showed_up",
    maxIter=10,
    weightCol="weightCol"
)


#### Recursive Feature Elimination

In [0]:
import os, sys, yaml

config_file = "../config.yaml" if os.path.exists("../config.yaml") else "config.example.yaml"
with open(config_file) as f:
    cfg = yaml.safe_load(f)

sys.path.insert(0, cfg["features_path"])
from engineering import recursive_feature_elimination

# Select N features to keep for RFE
N_FEATURES = 7

# Feature names in the order they were assembled into "scaledFeatures"
numerical_cols = [
    "Age", "Scholarship", "Hypertension", "Diabetes",
    "Alcoholism", "Handicap", "SMS_received", "date_diff",
    "Sched_month_sin", "Sched_month_cos",
    "Sched_dayofyear_sin", "Sched_dayofyear_cos",
    "Appoi_month_sin", "Appoi_month_cos",
    "Appoi_dayofyear_sin", "Appoi_dayofyear_cos",
    "Neighborhood_te", "Gender_vec",
]

selected_features, X_train_rfe, X_test_rfe, y_train_rfe, y_test_rfe = recursive_feature_elimination(
    train_scaled, test_scaled, numerical_cols, n_features_to_select=N_FEATURES
)

In [0]:
from pyspark.ml.feature import VectorSlicer

# Map selected feature names to their indices in the scaledFeatures vector
# (order matches numerical_cols in the RFE cell above)
feature_to_index = {name: idx for idx, name in enumerate(numerical_cols)}
selected_indices = [feature_to_index[f] for f in selected_features]
print(f"Selected feature indices: {selected_indices}")
print(f"Selected features: {selected_features}")

# Slice the scaledFeatures vector to keep only RFE-selected features
slicer = VectorSlicer(
    inputCol="scaledFeatures",
    outputCol="selectedFeatures",
    indices=selected_indices,
)

train_scaled = slicer.transform(train_scaled)
val_scaled = slicer.transform(val_scaled)
test_scaled = slicer.transform(test_scaled)

print(f"train_scaled now has selectedFeatures with {len(selected_indices)} features")

In [0]:
import math
from pyspark.sql.functions import when, col

# Recompute class weights using sqrt ratio for a milder correction
# (original 3.94x was too aggressive, pushing accuracy below the majority-class baseline)
no_show_count = train_scaled.filter(col("Showed_up") == 0).count()
show_count = train_scaled.filter(col("Showed_up") == 1).count()

WEIGHT_NO_SHOW = math.sqrt(show_count / no_show_count)
print(f"Adjusted weights — Showed_up=1: 1.0, Showed_up=0: {WEIGHT_NO_SHOW:.4f} (was {show_count / no_show_count:.4f})")

train_scaled = train_scaled.withColumn(
    "weightCol",
    when(col("Showed_up") == 1, 1.0).otherwise(WEIGHT_NO_SHOW)
)

#### Hyperparameter tuning

In [0]:
import numpy as np
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator, CrossValidatorModel

lr_param_grid = (ParamGridBuilder()
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
    .addGrid(lr.regParam, [0.001, 0.01, 0.05, 0.1, 0.5])
    .addGrid(lr.maxIter, [100, 500, 1000])
    .build()
) # End lr_param_grid

In [0]:
# Scorer
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol = "Showed_up",
    metricName = "areaUnderROC"
)

In [0]:
lr_cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=lr_param_grid,
    evaluator=evaluator,
    parallelism=6,
)

# Call CrossValidator on the training data
cvModel = lr_cv.fit(
    train_scaled
)  # Fits a model to the input dataset with optional parameters.
cvModel.getNumFolds()  # Gets the value of numFolds or its default value.

In [0]:
# Scorer
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol = "Showed_up",
    metricName = "areaUnderROC"
)

#### CrossValidator

In [0]:
# Use the cvModel already fitted in the CrossValidator cell above
lr_model = cvModel.bestModel
lr_val_predictions = lr_model.transform(val_scaled)
lr_predictions = lr_model.transform(test_scaled)

In [0]:
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)
from pyspark.sql import functions as F
import mlflow  # Experiment tracking & Record ML runs
import mlflow.spark  # Log model artifacts
import os
from mlflow.models import infer_signature

os.environ["MLFLOW_DFS_TMP"] = volume_path

In [0]:
#  columns: prediction, label, weight (optional) and probabilityCol (only for logLoss)
multi_evaluator = MulticlassClassificationEvaluator(
    predictionCol = 'prediction',
    labelCol = 'Showed_up'
)

#### Evaluation of LR

In [0]:
evaluator = BinaryClassificationEvaluator(
  rawPredictionCol = 'rawPrediction',
  labelCol = 'Showed_up',
  metricName = 'areaUnderROC')
  
auc = evaluator.evaluate(lr_predictions)

In [0]:
# print(multi_evaluator.explainParams())
recall = multi_evaluator.evaluate(lr_predictions, {multi_evaluator.metricName: "weightedRecall"})
f1 = multi_evaluator.evaluate(lr_predictions, {multi_evaluator.metricName: "f1"})
accuracy = multi_evaluator.evaluate(lr_predictions, {multi_evaluator.metricName: "accuracy"})
precision = multi_evaluator.evaluate(lr_predictions, {multi_evaluator.metricName: "weightedPrecision"})

#### Precision-Recall AUC

PR-AUC (Average Precision) summarizes the precision-recall curve into a single threshold-independent metric. Unlike ROC-AUC, it is sensitive to class imbalance and is more informative when the positive class is rare or when the business cost of false positives vs. false negatives is asymmetric.

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
import matplotlib.pyplot as plt
import numpy as np

# --- PR-AUC (threshold-independent, Spark-native) ---
pr_evaluator = BinaryClassificationEvaluator(
    labelCol="Showed_up",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)
pr_auc = pr_evaluator.evaluate(lr_predictions)
print(f"PR-AUC (Spark evaluator): {pr_auc:.4f}")

# --- PR Curve visualization ---
# Extract positive-class probability for plotting
get_prob_1 = F.udf(lambda v: float(v[1]), DoubleType())
lr_predictions = lr_predictions.withColumn("prob_1", get_prob_1("probability"))
lr_val_predictions = lr_val_predictions.withColumn("prob_1", get_prob_1("probability"))

# Collect label + score for plotting (test set only)
pdf = lr_predictions.select("Showed_up", "prob_1").toPandas()
y_true = pdf["Showed_up"].values.astype(int)
y_scores = pdf["prob_1"].values

# Sweep thresholds to build PR curve
thresholds = np.linspace(0.0, 1.0, 101)
precisions, recalls = [], []
for t in thresholds:
    y_pred = (y_scores >= t).astype(int)
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    p = tp / (tp + fp) if (tp + fp) > 0 else 1.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precisions.append(p)
    recalls.append(r)

# Trapezoidal sanity check against Spark's PR-AUC (sort by recall for correct sign)
order = np.argsort(recalls)
trapz_pr_auc = float(np.trapezoid(np.array(precisions)[order], np.array(recalls)[order]))
print(f"PR-AUC (trapezoidal):     {trapz_pr_auc:.4f}")

# Plot PR curve
fig, ax = plt.subplots(figsize=(7, 5))
ax.step(recalls, precisions, where='post', color='steelblue', linewidth=2)
ax.fill_between(recalls, precisions, step='post', alpha=0.2, color='steelblue')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title(f'Precision-Recall Curve (PR-AUC = {pr_auc:.4f})')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
display(fig)

#### Threshold Tuning

The default 0.5 decision threshold rarely maximizes F1 on imbalanced data. We sweep thresholds from 0.30 to 0.70 and pick the one with the best weighted F1, then recompute all metrics at that threshold.

In [0]:
from pyspark.sql import functions as F

# --- Threshold tuning on VALIDATION set ---
# Sweep thresholds on validation predictions to find the best F1.
# The test set is held out and only evaluated at the end with the tuned threshold.
best_threshold = 0.5
best_f1 = 0.0
print("Threshold sweep on VALIDATION set (weighted F1):")
for threshold in [i / 100.0 for i in range(30, 71, 5)]:
    temp_pred = lr_val_predictions.withColumn(
        "temp_prediction",
        F.when(F.col("prob_1") >= threshold, 1.0).otherwise(0.0)
    )
    temp_f1 = multi_evaluator.evaluate(
        temp_pred,
        {multi_evaluator.predictionCol: "temp_prediction",
         multi_evaluator.metricName: "f1"}
    )
    print(f"  threshold={threshold:.2f}  F1={temp_f1:.4f}")
    if temp_f1 > best_f1:
        best_f1 = temp_f1
        best_threshold = threshold

print(f"\nBest threshold (from validation): {best_threshold:.2f} (F1={best_f1:.4f})")

# Apply best threshold to the TEST set
lr_predictions = lr_predictions.withColumn(
    "prediction",
    F.when(F.col("prob_1") >= best_threshold, 1.0).otherwise(0.0)
)

# --- Weighted metrics on TEST set (tuned threshold) ---
precision = multi_evaluator.evaluate(lr_predictions, {multi_evaluator.metricName: "weightedPrecision"})
recall = multi_evaluator.evaluate(lr_predictions, {multi_evaluator.metricName: "weightedRecall"})
f1 = multi_evaluator.evaluate(lr_predictions, {multi_evaluator.metricName: "f1"})
accuracy = multi_evaluator.evaluate(lr_predictions, {multi_evaluator.metricName: "accuracy"})

# --- Per-class metrics ---
confusion = lr_predictions.groupBy("Showed_up", "prediction").count().collect()
cm = {}
for row in confusion:
    cm[(float(row['Showed_up']), float(row['prediction']))] = row['count']

def class_metrics(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_val = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f1_val

# Class 1 (Showed up)
p_1, r_1, f1_1 = class_metrics(
    cm.get((1.0, 1.0), 0), cm.get((0.0, 1.0), 0), cm.get((1.0, 0.0), 0)
)
# Class 0 (No-show)
p_0, r_0, f1_0 = class_metrics(
    cm.get((0.0, 0.0), 0), cm.get((1.0, 0.0), 0), cm.get((0.0, 1.0), 0)
)

print(f"\nNo-show class recall (Class 0): {r_0:.4f}")

In [0]:
signature_lr = infer_signature(train_scaled, lr_predictions)

In [0]:
mlflow.end_run()  # Clean up any stale active run
mlflow.set_experiment(cfg["mlflow"]["experiment_path"])

with mlflow.start_run(run_name="LogisticRegressionModel") as run:
    mlflow.spark.log_model(lr_model, "LogisticRegressionModel") # skipped — Serverless 256MB limit
    mlflow.log_param("maxIter", 10)
    mlflow.log_param("featuresCol", "features")
    mlflow.log_param("labelCol", "Showed_up")
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("best_threshold", best_threshold)
    mlflow.log_metric("f1_class1_showed_up", f1_1)
    mlflow.log_metric("f1_class0_no_show", f1_0)

### Log model run

In [0]:
# Register the model with signature
model_info = mlflow.spark.log_model(
    lr_model,
    "LogisticRegressionModel",
    signature=signature_lr,
    registered_model_name= logistic_regression_model_path
)

print(f"Registered model version: {model_info.registered_model_version}")

In [0]:
from mlflow import MlflowClient

client = MlflowClient()
client.set_registered_model_alias(
    name=logistic_regression_model_path,
    alias="champion",
    version=model_info.registered_model_version
)

In [0]:
print("=" * 50)
print("  Logistic Regression — Run Summary")
print("=" * 50)
print()
print("RFE Selected Features:")
for f in selected_features:
    print(f"  • {f}")
print()
print(f"Best regParam:        {lr_model.getRegParam()}")
print(f"Best maxIter:         {lr_model.getMaxIter()}")
print(f"Best threshold:       {best_threshold:.2f}")
print()
print("Evaluation Metrics (test set, tuned threshold):")
print(f"  ROC-AUC:   {auc:.4f}")
print(f"  PR-AUC:    {pr_auc:.4f}")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  F1:        {f1:.4f}  (weighted)")
print(f"  Precision: {precision:.4f}  (weighted)")
print(f"  Recall:    {recall:.4f}  (weighted)")
print(f"  No-show class recall (Class 0): {r_0:.4f}")
print()
print("Per-class Metrics:")
print(f"  Class 1 (Showed up):  P={p_1:.4f}  R={r_1:.4f}  F1={f1_1:.4f}")
print(f"  Class 0 (No-show):    P={p_0:.4f}  R={r_0:.4f}  F1={f1_0:.4f}")
print()
print(f"Registered model:    {logistic_regression_model_path}")
print(f"Model version:       {model_info.registered_model_version}")
print(f"Alias:               champion")
print("=" * 50)